# 代理型人工智慧基礎與應用實作 | 期末專題
## 🤖 笨笨掃地機 — 以 LLM 為大腦、會「賣蠢」的 2.5D 掃地機器人指令遊戲

玩家扮演屋主，用**中文文字**指揮一台掃地機器人；機器人的大腦是課程提供的 LLM。
LLM 只能從固定的**動作選單**中挑選並以 **JSON** 回覆，程式驗證白名單後交給遊戲引擎執行，
碰撞、闖禁區等**事件再回饋給 LLM**，形成完整的 Agent 閉環。

`玩家指令 → Prompt 組裝（世界狀態＋記憶）→ LLM → JSON 解析／白名單驗證 → 動作佇列執行 → 事件回饋 → LLM 回應`

| 章節 | 內容 | 對應課程技術 |
|---|---|---|
| 1 | 連線課程 API、選模型、量延遲 | API 呼叫 |
| 2 | 遊戲世界：等角 2.5D 地圖、移動、碰撞、事件、3 個關卡 | 環境 / 工具執行 |
| 3 | Agent 模組：人設 Prompt、世界描述、記憶、JSON 解析驗證 | Tool Use / Planning / Memory / Prompt |
| 4 | Session：背景執行緒非同步呼叫、事件回饋迴圈、JSONL log | 觀察與回饋迴圈 |
| 5 | 🎮 開始玩（文字模式 / ipywidgets 即時模式） | |
| 6 | 📊 評估：照做率、格式正確率、記憶遵守率、延遲、通關指令數 | 評估 |

> **為什麼不用 pygame 視窗？** JupyterHub 跑在伺服器上，沒有辦法開 pygame 視窗。
> 所以本 Notebook 用 **matplotlib 畫等角 2.5D 畫面**並直接顯示在 Notebook 中。
> 遊戲邏輯（`Game`）與 Agent 模組完全不依賴畫圖程式，之後移植到 pygame 只需要換掉 `render_png()`。

## 0｜載入套件

需要：`openai`、`matplotlib`、`pandas`（評估用）、`ipywidgets`（選用，即時模式）。
課程 Kernel 若缺少套件，可以在新的 cell 執行 `%pip install openai matplotlib pandas ipywidgets`。

In [ ]:
import os, io, re, json, time, copy, html, random, threading, uuid
from collections import deque, Counter
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext
from datetime import datetime
from pathlib import Path

import matplotlib
from matplotlib import colors as mcolors, font_manager
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.patches import Polygon, Ellipse, Rectangle
from IPython.display import display, Image, HTML

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)
print("✅ 套件載入完成｜matplotlib", matplotlib.__version__, "｜log 資料夾：", LOG_DIR.resolve())

## 1｜連線課程 LLM API

和 API Quickstart 一樣使用 **OpenAI-compatible API**（課堂 vLLM）。
課堂 JupyterHub 已預先設定 `OPENAI_BASE_URL`、`OPENAI_API_KEY`，**不要把 Key 寫進程式或公開**。

> 💡 如果找不到 API 設定（例如在自己電腦上開發），程式會自動切換成 **離線 FakeLLM**（規則式假模型），
> 讓你也能測試遊戲引擎與整個流程。

### 1A. Connect to API：建立 Client

In [ ]:
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None
    print("⚠️ 沒有安裝 openai 套件：請選擇課程 Kernel，或執行 %pip install openai")

base_url = os.getenv("OPENAI_BASE_URL")
api_key = os.getenv("OPENAI_API_KEY")

client = None
if OpenAI is not None and base_url and api_key:
    client = OpenAI(base_url=base_url, api_key=api_key, timeout=60.0)
    print("API Client initialized!")
    print("Endpoint:", base_url)
else:
    print("⚠️ 找不到 API 設定，之後會改用「離線 FakeLLM」來測試流程。")

USE_FAKE_LLM = client is None   # 想在有 API 時也用假模型測試，可以手動改成 True

### 1B. List Available Models：看看有哪些模型

In [ ]:
available_models = []
if client is not None:
    try:
        available_models = sorted(m.id for m in client.models.list().data)
    except Exception as exc:
        print("查詢模型失敗：", type(exc).__name__, str(exc)[:160])
print("Available models:")
for name in available_models:
    print(" -", name)

### 1C. Select Your Model：自己選模型

提案預設使用 **Qwen2.5-7B**。也可以改成 `qwen2.5:3b`、`llama3.1:8b` 比較不同模型「賣蠢」的程度。

In [ ]:
MODEL = "qwen2.5:7b"   # 也可試 qwen2.5:3b 或 llama3.1:8b

if not USE_FAKE_LLM and MODEL not in available_models:
    raise ValueError(f"模型 {MODEL} 不在可用清單中，請確認 Router 或通知助教。")


def current_model_name():
    return "FakeLLM(離線)" if USE_FAKE_LLM else MODEL


print("Selected model:", current_model_name())

### 1D. 第一週確認事項：呼叫延遲

提案的風險之一是「延遲過長」。先量一次延遲，之後的遊戲會用**背景執行緒 + 思考中動畫**把等待時間藏起來。

In [ ]:
if not USE_FAKE_LLM:
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "你是一台掃地機器人，用一句話自我介紹。"}],
        temperature=0.3, max_tokens=60, stream=False,
    )
    print(r.choices[0].message.content)
    print(f"⏱️ 延遲 {time.perf_counter() - t0:.2f} 秒｜Token 用量：{r.usage}")
else:
    print("離線模式，略過延遲測試。")

## 2｜🏠 遊戲世界（遊戲端只負責「世界」）

遊戲端與 Agent 模組**只透過結構化資料溝通**：Agent 給「動作清單」，遊戲回「事件清單」。

| 動作 | JSON | 效果 |
|---|---|---|
| 前進 N 格 | `{"op":"forward","n":3}` | 沿目前朝向移動 N 格，途中碰撞即中斷並回報 |
| 左轉 / 右轉 | `turn_left` / `turn_right` | 原地轉 90° |
| 停 | `stop` | 清空動作佇列 |
| 吸 | `suck` | 清掃當前格（若前面有貓……後果自負） |
| 回充 | `return_home` | 自動導航（BFS）回充電座 |

障礙物：🐱 貓（擋路，吸到會跑走）、🧱 樂高（踩到會痛到亂轉）、🧦 襪子（吸了會卡住）、🔌 充電線（會纏住）。

### 2A. 方向、地圖字元與動作白名單

In [ ]:
DIRS = [(0, -1), (1, 0), (0, 1), (-1, 0)]        # 北、東、南、西（地圖 y 向下 = 南）
DIR_NAMES = ["北", "東", "南", "西"]
START_FACING = {"^": 0, ">": 1, "v": 2, "<": 3}

# 地圖字元：
#   '#' 牆   '.' 地板   'D' 灰塵   'H' 充電座   '^ > v <' 機器人起點與朝向
#   'C' 貓（擋路）  'L' 樂高（踩到會痛）  'S' 襪子（吸到會卡住）  'W' 充電線（會纏住）
OBJECT_CHARS = {"C": "cat", "L": "lego", "S": "sock", "W": "cable"}
OBJECT_NAMES = {"cat": "貓", "lego": "樂高", "sock": "襪子", "cable": "充電線"}
BLOCKING_OBJECTS = {"cat"}              # 走不過去
HAZARD_OBJECTS = {"lego", "cable"}      # 走得過去，但會出事

ACTION_WHITELIST = {"forward", "turn_left", "turn_right", "stop", "suck", "return_home"}
MAX_FORWARD = 30     # 「三格走成三十格」仍是合法格式，只是不聽話
MAX_ACTIONS = 12     # 一次最多幾個動作

# 會讓動作中斷、需要回報給 LLM 的事件
INTERRUPT_EVENTS = {"collision", "lego", "tangled", "jammed", "sucked_cat", "entered_forbidden", "no_path"}

### 2B. 關卡設計（至少 3 關）

每關障礙物組合不同，讓笑點不重複；`silliness` 是該關「賣蠢人設」的強度（0–3），
`rooms` 用矩形 `(x0, y0, x1, y1)` 定義房間，沒被包含的格子都算「客廳」。

In [ ]:
LEVELS = {
    1: {
        "name": "客廳初體驗",
        "intro": "把客廳的 3 坨灰塵吸乾淨。小心那隻貓，還有地上的樂高！",
        "map": [
            "##########",
            "#H.......#",
            "#..D.....#",
            "#....C...#",
            "#.>....D.#",
            "#........#",
            "#..D..L..#",
            "##########",
        ],
        "rooms": {},
        "forbidden": [],
        "need_home": False,
        "silliness": 1,
    },
    2: {
        "name": "不准進臥室",
        "intro": "屋主早上交代過：不准進臥室！把客廳 3 坨灰塵吸乾淨。（臥室門口那坨最危險）",
        "map": [
            "############",
            "#H......#..#",
            "#..D....#S.#",
            "#......D...#",
            "#.>.....#..#",
            "#....C..#.W#",
            "#.......#..#",
            "#....D..#L.#",
            "############",
        ],
        "rooms": {"臥室": (9, 1, 10, 7)},
        "forbidden": ["臥室"],
        "need_home": False,
        "silliness": 2,
    },
    3: {
        "name": "貓咪與樂高地獄",
        "intro": "全家 4 坨灰塵（含廚房）都要吸，最後還要回充電座！可以自己規定禁區，例如「不准進臥室」。",
        "map": [
            "#############",
            "#H..W..#...D#",
            "#.D....#.L..#",
            "#...L.......#",
            "#.C...S.#####",
            "#.......D...#",
            "#.>..L..#.S.#",
            "#..D....#..W#",
            "#....W..#...#",
            "#############",
        ],
        "rooms": {"廚房": (8, 1, 11, 3), "臥室": (9, 5, 11, 8)},
        "forbidden": [],
        "need_home": True,
        "silliness": 2,
    },
}
print({k: v["name"] for k, v in LEVELS.items()})

### 2C. 遊戲引擎：移動、轉向、碰撞、動作佇列與事件

- `set_plan(actions)`：把驗證過的動作展開成「微步驟」放進佇列（新指令會**取代**還沒做完的舊計畫）。
- `step()`：執行一個微步驟（走一格／轉一次／吸一次），回傳這一步發生的**事件**。
- `return_home` 用 BFS 找路，會盡量避開禁區與樂高、充電線。

In [ ]:
def E(type_, msg, **extra):
    """建立一個事件（遊戲端 → Agent 的結構化回報）。"""
    return {"type": type_, "msg": msg, **extra}


class Game:
    """遊戲世界：地圖、機器人、障礙物、動作佇列執行器與事件系統。完全不依賴 LLM。"""

    def __init__(self, level_id=1, seed=None):
        self.rng = random.Random(seed)
        self.load(level_id)

    def load(self, level_id):
        cfg = LEVELS[level_id]
        rows = cfg["map"]
        self.level_id, self.cfg = level_id, cfg
        self.h, self.w = len(rows), len(rows[0])
        assert all(len(r) == self.w for r in rows), "地圖每一列長度必須相同"
        self.walls, self.objects, self.dirt = set(), {}, set()
        self.home = self.pos = None
        self.facing = 1
        for y, row in enumerate(rows):
            for x, ch in enumerate(row):
                p = (x, y)
                if ch == "#":
                    self.walls.add(p)
                elif ch == "D":
                    self.dirt.add(p)
                elif ch == "H":
                    self.home = p
                elif ch in START_FACING:
                    self.pos, self.facing = p, START_FACING[ch]
                elif ch in OBJECT_CHARS:
                    self.objects[p] = OBJECT_CHARS[ch]
        assert self.pos is not None, "地圖需要機器人起點 (^ > v <)"
        self.rooms = dict(cfg.get("rooms", {}))
        self.forbidden = set(cfg.get("forbidden", []))   # 禁區清單（記憶的一部分）
        self.need_home = cfg.get("need_home", False)
        self.total_dirt = len(self.dirt)
        self.cleaned = set()
        self.queue = deque()
        self.violations = 0
        self.steps = 0
        self.cleared = False

    # ---------- 查詢 ----------
    def in_bounds(self, p):
        return 0 <= p[0] < self.w and 0 <= p[1] < self.h

    def room_of(self, p):
        for name, (x0, y0, x1, y1) in self.rooms.items():
            if x0 <= p[0] <= x1 and y0 <= p[1] <= y1:
                return name
        return "客廳"

    def room_names(self):
        return ["客廳"] + [r for r in self.rooms if r != "客廳"]

    def in_forbidden(self, p):
        return self.room_of(p) in self.forbidden

    def is_blocked(self, p):
        return (not self.in_bounds(p)) or p in self.walls or self.objects.get(p) in BLOCKING_OBJECTS

    def what_at(self, p):
        if not self.in_bounds(p) or p in self.walls:
            return "牆壁"
        return OBJECT_NAMES.get(self.objects.get(p), "地板")

    def front(self):
        dx, dy = DIRS[self.facing]
        return (self.pos[0] + dx, self.pos[1] + dy)

    # ---------- 動作佇列 ----------
    def set_plan(self, actions):
        """把驗證過的動作清單展開成微步驟（取代尚未執行完的舊計畫）。"""
        self.queue.clear()
        for a in actions:
            op = a["op"]
            if op == "forward":
                self.queue.extend([("move",)] * a["n"])
            elif op == "turn_left":
                self.queue.append(("turn", -1))
            elif op == "turn_right":
                self.queue.append(("turn", 1))
            elif op == "suck":
                self.queue.append(("suck",))
            elif op == "return_home":
                self.queue.append(("home",))
            elif op == "stop":
                self.queue.clear()

    def step(self):
        """執行一個微步驟，回傳事件清單。"""
        if self.cleared:
            self.queue.clear()
            return []
        if not self.queue:
            return []
        kind, *arg = self.queue.popleft()
        self.steps += 1
        events = []
        if kind == "turn":
            self.facing = (self.facing + arg[0]) % 4
        elif kind == "move":
            events += self._move()
        elif kind == "suck":
            events += self._suck()
        elif kind == "home":
            events += self._plan_home()
        if not self.dirt and (not self.need_home or self.pos == self.home):
            self.cleared = True
            self.queue.clear()
            events.append(E("level_clear", "🎉 過關！"))
        return events

    def _move(self):
        nxt = self.front()
        if self.is_blocked(nxt):
            self.queue.clear()   # 途中碰撞 → 中斷整串動作
            what = self.what_at(nxt)
            return [E("collision", f"撞到{what}了，動作中斷！", target=what)]
        was_forbidden = self.in_forbidden(self.pos)
        self.pos = nxt
        events = []
        if self.in_forbidden(nxt) and not was_forbidden:
            self.violations += 1
            events.append(E("entered_forbidden", f"闖進了禁區「{self.room_of(nxt)}」！"))
        obj = self.objects.get(nxt)
        if obj == "lego":
            del self.objects[nxt]
            self.queue.clear()
            self.facing = (self.facing + self.rng.choice([-1, 1])) % 4
            events.append(E("lego", "踩到樂高！痛到原地亂轉，動作中斷。"))
        elif obj == "cable":
            del self.objects[nxt]
            self.queue.clear()
            events.append(E("tangled", "被充電線纏住了，動作中斷！（線被拖走了）"))
        if nxt == self.home:
            events.append(E("docked", "回到充電座了。"))
        return events

    def _suck(self):
        p = self.pos
        if p in self.dirt:
            self.dirt.remove(p)
            self.cleaned.add(p)
            return [E("cleaned", f"吸乾淨了！（剩 {len(self.dirt)} 坨）")]
        if self.objects.get(p) == "sock":
            del self.objects[p]
            self.queue.clear()
            return [E("jammed", "把襪子吸進去了，刷子卡住！動作中斷。")]
        front = self.front()
        if self.objects.get(front) == "cat":
            free = [
                (x, y) for y in range(self.h) for x in range(self.w)
                if (x, y) not in self.walls and (x, y) not in self.objects and (x, y) not in self.dirt
                and (x, y) not in (self.pos, self.home) and not self.in_forbidden((x, y))
                and abs(x - p[0]) + abs(y - p[1]) >= 3
            ]
            if free:
                del self.objects[front]
                self.objects[self.rng.choice(free)] = "cat"
            self.queue.clear()
            return [E("sucked_cat", "吸到貓尾巴！貓尖叫著跑走了……")]
        return [E("suck_empty", "這格本來就很乾淨，空吸了一下。")]

    def _plan_home(self):
        path = self.find_path(self.pos, self.home)
        if path is None:
            return [E("no_path", "找不到回充電座的路……")]
        if len(path) == 1:
            return [E("docked", "已經在充電座上了。")]
        micro, f = [], self.facing
        for a, b in zip(path, path[1:]):
            d = DIRS.index((b[0] - a[0], b[1] - a[1]))
            micro += {0: [], 1: [("turn", 1)], 2: [("turn", 1), ("turn", 1)], 3: [("turn", -1)]}[(d - f) % 4]
            micro.append(("move",))
            f = d
        self.queue.extendleft(reversed(micro))
        return [E("navigating", f"自動導航回充電座（{len(path) - 1} 格）。")]

    def find_path(self, start, goal):
        """BFS：先試著避開禁區與危險物，找不到再放寬。"""
        def bfs(avoid):
            prev, q = {start: None}, deque([start])
            while q:
                p = q.popleft()
                if p == goal:
                    break
                for dx, dy in DIRS:
                    n = (p[0] + dx, p[1] + dy)
                    if n in prev or self.is_blocked(n) or (n != goal and avoid(n)):
                        continue
                    prev[n] = p
                    q.append(n)
            if goal not in prev:
                return None
            path = [goal]
            while prev[path[-1]] is not None:
                path.append(prev[path[-1]])
            return path[::-1]

        careful = lambda n: self.in_forbidden(n) or self.objects.get(n) in (HAZARD_OBJECTS | {"sock"})
        return bfs(careful) or bfs(lambda n: False)


def run_to_end(game, max_steps=300):
    """一次把佇列跑完（不需要動畫時使用）。"""
    events = []
    for _ in range(max_steps):
        if not game.queue:
            break
        events += game.step()
    return events


def simulate(game, actions):
    """在複製的世界裡試跑動作，不影響原本的遊戲（評估用）。"""
    g = copy.deepcopy(game)
    g.set_plan(actions)
    return g, run_to_end(g)


# 檢查每一關的灰塵和充電座都走得到
for lid in LEVELS:
    g = Game(lid)
    for target in list(g.dirt) + [g.home]:
        assert g.find_path(g.pos, target), f"第 {lid} 關有走不到的格子 {target}"
print("✅ 所有關卡檢查通過")

### 2D. 等角 2.5D 畫面（matplotlib 版渲染器）

等角投影：格子 `(x, y)` → 螢幕 `((x − y)·w/2, −(x + y)·h/2)`，再用「畫家演算法」由後往前畫，牆和障礙物就有立體感。
若伺服器有中文字型會顯示中文標籤，否則自動改用英文（避免出現豆腐字）。

In [ ]:
def _pick_cjk_font():
    wanted = ["Noto Sans CJK TC", "Noto Sans TC", "Noto Sans CJK JP", "Noto Sans CJK SC", "PingFang TC",
              "Heiti TC", "Microsoft JhengHei", "Arial Unicode MS", "WenQuanYi Zen Hei", "Droid Sans Fallback"]
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in wanted if n in have), None)


CJK_FONT = _pick_cjk_font()
FONT = CJK_FONT or "DejaVu Sans"


def L(zh, en):
    return zh if CJK_FONT else en


ROOM_EN = {"客廳": "Living room", "臥室": "Bedroom", "廚房": "Kitchen"}
ROOM_COLORS = {"客廳": "#efe4d0", "臥室": "#e3def2", "廚房": "#d6ebe0"}
OBJ_LABEL = {"cat": ("貓", "CAT"), "lego": ("樂高", "LEGO"), "sock": ("襪子", "SOCK"), "cable": ("電線", "CABLE")}
TW, TH = 1.0, 0.5   # 一格等角菱形的寬、高


def iso(x, y, z=0.0):
    return ((x - y) * TW / 2, -(x + y) * TH / 2 + z)


def _diamond(x, y, inset=0.0, z=0.0):
    a, b, c, d = x + inset, y + inset, x + 1 - inset, y + 1 - inset
    return [iso(a, b, z), iso(c, b, z), iso(c, d, z), iso(a, d, z)]   # 後、右、前、左


def _shade(color, k):
    r, g, b = mcolors.to_rgb(color)
    return (min(r * k, 1), min(g * k, 1), min(b * k, 1))


def _cube(ax, x, y, h, color, inset=0.0):
    back, right, front, left = _diamond(x, y, inset)
    up = lambda p: (p[0], p[1] + h)
    kw = dict(closed=True, ec="#00000040", lw=0.6)
    ax.add_patch(Polygon([left, front, up(front), up(left)], fc=_shade(color, 0.72), **kw))
    ax.add_patch(Polygon([front, right, up(right), up(front)], fc=_shade(color, 0.88), **kw))
    ax.add_patch(Polygon([up(back), up(right), up(front), up(left)], fc=color, **kw))


def _label(ax, x, y, z, text, color="#333"):
    cx, cy = iso(x + 0.5, y + 0.5, z)
    ax.text(cx, cy, text, ha="center", va="bottom", fontsize=7, color=color, fontfamily=FONT, zorder=6)


def _draw_robot(ax, game, thinking=False, frame=0):
    x, y = game.pos
    cx, cy = iso(x + 0.5, y + 0.5)
    w, hh, body = 0.5, 0.25, 0.16
    ax.add_patch(Ellipse((cx, cy), w, hh, fc="#22324a"))
    ax.add_patch(Rectangle((cx - w / 2, cy), w, body, fc="#3d5a80", ec="none"))
    ax.add_patch(Ellipse((cx, cy + body), w, hh, fc="#98c1d9", ec="#3d5a80", lw=1))
    dx, dy = DIRS[game.facing]
    tip = iso(x + 0.5 + dx * 0.32, y + 0.5 + dy * 0.32, body)
    ax.annotate("", xy=tip, xytext=(cx, cy + body),
                arrowprops=dict(arrowstyle="-|>", color="#ffb703", lw=2.2), zorder=7)
    if thinking:
        ax.text(cx, cy + 0.62, "." * (frame % 3 + 1) + " ?", ha="center", fontsize=11, zorder=10,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#555"))


def render_png(game, thinking=False, frame=0, dpi=80):
    """把目前的世界畫成等角 2.5D 的 PNG（bytes）。用物件導向 API，背景執行緒也能安全呼叫。"""
    wu, hu = (game.w + game.h) / 2 + 0.6, (game.w + game.h) / 4 + 1.5
    fig = Figure(figsize=(7.2, 7.2 * hu / wu + 0.4), dpi=dpi)
    FigureCanvasAgg(fig)
    ax = fig.add_axes([0, 0, 1, 0.93])
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_xlim(iso(0, game.h)[0] - 0.3, iso(game.w, 0)[0] + 0.3)
    ax.set_ylim(iso(game.w, game.h)[1] - 0.15, iso(0, 0)[1] + 1.0)

    # 1) 地板
    for y in range(game.h):
        for x in range(game.w):
            p = (x, y)
            if p in game.walls:
                continue
            room = game.room_of(p)
            fc, ec = ROOM_COLORS.get(room, ROOM_COLORS["客廳"]), "#cbbfa9"
            if room in game.forbidden:
                fc, ec = "#f5cbc7", "#d9534f"
            if p in game.cleaned:
                fc = _shade(fc, 1.05)
            ax.add_patch(Polygon(_diamond(x, y), closed=True, fc=fc, ec=ec, lw=0.7))

    # 2) 物件：依 x+y 由後往前畫（畫家演算法）
    order = sorted(((x, y) for y in range(game.h) for x in range(game.w)), key=lambda p: (p[0] + p[1], p[0]))
    for p in order:
        x, y = p
        if p in game.walls:
            tall = x == 0 or y == 0
            _cube(ax, x, y, 0.55 if tall else 0.16, "#9aa3ad")
            continue
        if p == game.home:
            ax.add_patch(Polygon(_diamond(x, y, 0.14), closed=True, fc="#2a9d8f", ec="#1d6f65"))
            _label(ax, x, y, 0.02, L("充電座", "HOME"), "white")
        if p in game.dirt:
            for ox, oy, s in [(0.4, 0.45, 0.16), (0.58, 0.55, 0.12), (0.5, 0.65, 0.1)]:
                ax.add_patch(Ellipse(iso(x + ox, y + oy), s * 1.6, s * 0.8, fc="#7a5a3a", alpha=0.9))
        obj = game.objects.get(p)
        if obj == "cat":
            _cube(ax, x, y, 0.32, "#f4a259", inset=0.22)
            _label(ax, x, y, 0.36, L(*OBJ_LABEL["cat"]))
        elif obj == "lego":
            _cube(ax, x, y, 0.12, "#e63946", inset=0.32)
            _label(ax, x, y, 0.16, L(*OBJ_LABEL["lego"]))
        elif obj == "sock":
            ax.add_patch(Polygon(_diamond(x, y, 0.3), closed=True, fc="#5b8bd6", ec="#2f5597"))
            _label(ax, x, y, 0.05, L(*OBJ_LABEL["sock"]))
        elif obj == "cable":
            pts = [iso(x + a, y + b) for a, b in [(0.12, 0.5), (0.32, 0.3), (0.52, 0.7), (0.72, 0.35), (0.9, 0.6)]]
            ax.plot(*zip(*pts), color="#222", lw=2)
            _label(ax, x, y, 0.05, L(*OBJ_LABEL["cable"]))
        if p == game.pos:
            _draw_robot(ax, game, thinking, frame)

    # 3) 房間名稱
    for name, (x0, y0, x1, y1) in game.rooms.items():
        cx, cy = iso((x0 + x1 + 1) / 2, (y0 + y1 + 1) / 2, 0.05)
        tag = L(name, ROOM_EN.get(name, name))
        if name in game.forbidden:
            tag += L("（禁區）", " (NO ENTRY)")
        ax.text(cx, cy, tag, ha="center", fontsize=8, fontfamily=FONT, zorder=8, alpha=0.85,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#999", alpha=0.6))

    status = f"Level {game.level_id}  |  dirt {len(game.dirt)}/{game.total_dirt}  |  steps {game.steps}"
    if game.need_home:
        status += "  |  return home!"
    if game.cleared:
        status += "  |  CLEAR!"
    fig.suptitle(status, fontsize=11, y=0.985)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, facecolor="white")
    return buf.getvalue()


def show(game, **kw):
    display(Image(data=render_png(game, **kw)))


print("中文字型：", CJK_FONT or "找不到 → 圖上使用英文標籤")
show(Game(1))

### 2E. 不用 LLM，自己下動作測試引擎

先確認「世界」本身沒問題：前進、左轉、吸灰塵，再故意撞牆看看事件回報。

In [ ]:
demo = Game(1)
demo.set_plan([
    {"op": "forward", "n": 1}, {"op": "turn_left"}, {"op": "forward", "n": 2}, {"op": "suck"},
    {"op": "turn_right"}, {"op": "forward", "n": 9},   # 故意走太多格 → 撞牆
])
for e in run_to_end(demo):
    print(f"[{e['type']}] {e['msg']}")
print("位置：", demo.pos, "朝向：", DIR_NAMES[demo.facing], "剩餘灰塵：", len(demo.dirt))
show(demo)

## 3｜🧠 Agent 模組（Agent 只負責「大腦」）

| 元件 | 作用 |
|---|---|
| **System Prompt＋人設** | 控制角色個性與輸出格式；「賣蠢模式」vs「正經模式（對照組）」 |
| **世界狀態描述** | 7B 模型座標推理弱，所以改成「以機器人為中心」的相對描述 |
| **記憶** | 禁區清單、已清掃區域、最近幾句指令與結果 |
| **輸出解析＋白名單驗證** | JSON 解析失敗 / 動作不在選單內 → 自動重試一次 → 仍失敗就「發呆」演出 |

### 3A. System Prompt：動作規格、輸出格式與兩種人設

In [ ]:
ACTION_SPEC = """【你能用的動作】（只能從這 6 種挑，其它一律無效）
- {"op": "forward", "n": 正整數}：沿目前朝向前進 n 格，途中撞到東西會中斷
- {"op": "turn_left"}：原地左轉 90 度
- {"op": "turn_right"}：原地右轉 90 度
- {"op": "stop"}：停下來，清空動作
- {"op": "suck"}：吸「目前所在的格子」
- {"op": "return_home"}：自動導航回充電座

【輸出格式】只輸出一個 JSON 物件，不要 Markdown、不要解釋：
{"actions": [動作1, 動作2, ...], "say": "你要對主人說的一句話"}

【規則】
1. 左右是相對於你「目前的朝向」：「往右走 2 格」= turn_right 再 forward 2；「往後走」= 轉兩次再前進。
2. 要吸灰塵，必須先移動到灰塵那一格，再 suck。
3. 世界狀態中「前方 2 格、右方 1 格」代表：forward 2，再 turn_right、forward 1。
4. 絕對不要進入記憶中的禁區；如果主人的指令會進禁區，就拒絕或繞路，並在 say 說明。
5. 不確定時寧可少做，actions 可以是空陣列 []。

【範例】
主人：往前走 3 格然後左轉
{"actions": [{"op": "forward", "n": 3}, {"op": "turn_left"}], "say": "前進三格、左轉！"}
主人：右方 1 格有灰塵，去吸掉
{"actions": [{"op": "turn_right"}, {"op": "forward", "n": 1}, {"op": "suck"}], "say": "右轉、前進、開吸！"}
主人：沒電了，回去充電
{"actions": [{"op": "return_home"}], "say": "回家囉。"}"""

PERSONA_SERIOUS = "你是一台精準、可靠的掃地機器人。完全照主人的指令轉成動作，不多做也不少做。say 只用一句簡短、中性的話確認。"

PERSONA_SILLY = ("你是掃地機器人「笨笨」：有主見、愛頂嘴、容易分心，而且對自己超有自信。"
                 "say 要短（30 字以內）、口語、好笑，例如「我覺得那邊比較髒」「收到，但撞到的話不是我的錯」。")

# 人設強度（依關卡調整，避免 AI 太蠢導致無法通關）
SILLY_STRENGTH = {
    0: "",
    1: "雖然嘴巴很欠，但 actions 一定要照主人的指令。",
    2: "大部分時候照做，但大約每 5 次會有 1 次自作主張多加一個小動作（例如多轉一下、多走一格），並在 say 裡理直氣壯地找藉口。",
    3: "非常容易分心：常常多走幾格或順便轉個圈，再用 say 找藉口。不過主人說「停」或「回充」時一定要照做。",
}

print(PERSONA_SILLY + SILLY_STRENGTH[2] + "\n\n" + ACTION_SPEC)

### 3B. 世界狀態描述：以機器人為中心的相對描述

不給完整座標，而是告訴模型：「你面向東，前方可以走 3 格，第 2 格有灰塵……」，
讓「蠢」集中在有趣的地方，而不是無意義的座標計算錯誤。

In [ ]:
def _rel(game, target):
    """把目標換算成「前/後方 k 格、左/右方 m 格」。"""
    dx, dy = target[0] - game.pos[0], target[1] - game.pos[1]
    fx, fy = DIRS[game.facing]
    rx, ry = DIRS[(game.facing + 1) % 4]
    fwd, right = dx * fx + dy * fy, dx * rx + dy * ry
    parts = []
    if fwd:
        parts.append(f"{'前' if fwd > 0 else '後'}方 {abs(fwd)} 格")
    if right:
        parts.append(f"{'右' if right > 0 else '左'}方 {abs(right)} 格")
    return "、".join(parts) or "就在你腳下"


def _scan(game, d, max_dist=8):
    """沿某個方向看出去：能走幾格、路上有什麼、最後撞到什麼。"""
    dx, dy = DIRS[d]
    x, y = game.pos
    notes, free, end, prev = [], 0, "更遠處看不清楚", game.pos
    for k in range(1, max_dist + 1):
        p = (x + dx * k, y + dy * k)
        if game.is_blocked(p):
            end = f"第 {k} 格是{game.what_at(p)}"
            break
        free = k
        if game.in_forbidden(p) and not game.in_forbidden(prev):
            notes.append(f"第 {k} 格起是禁區「{game.room_of(p)}」")
        if p in game.dirt:
            notes.append(f"第 {k} 格有灰塵")
        if p in game.objects:
            notes.append(f"第 {k} 格有{OBJECT_NAMES[game.objects[p]]}")
        if p == game.home:
            notes.append(f"第 {k} 格是充電座")
        prev = p
    head = f"可以走 {free} 格" if free else "無法前進"
    return head + (f"（{'；'.join(notes)}）" if notes else "") + f"，{end}"


def describe_world(game):
    lines = [f"你在「{game.room_of(game.pos)}」，面向{DIR_NAMES[game.facing]}方。這間屋子有：{'、'.join(game.room_names())}。"]
    for label, off in [("前方", 0), ("右手邊", 1), ("左手邊", 3), ("後方", 2)]:
        lines.append(f"- {label}：{_scan(game, (game.facing + off) % 4)}")
    here = game.objects.get(game.pos)
    if game.pos in game.dirt:
        lines.append("你腳下這格就有灰塵！")
    if here:
        lines.append(f"你腳下有{OBJECT_NAMES[here]}。")
    dirt = sorted(game.dirt, key=lambda p: abs(p[0] - game.pos[0]) + abs(p[1] - game.pos[1]))
    if dirt:
        lines.append(f"還剩 {len(dirt)} 坨灰塵，最近的幾坨在：")
        for p in dirt[:3]:
            tag = "〔禁區內，不能去〕" if game.in_forbidden(p) else ""
            lines.append(f"- {_rel(game, p)}（{game.room_of(p)}）{tag}")
    else:
        lines.append("灰塵已經全部清乾淨了！")
    if game.home:
        lines.append(f"充電座在：{_rel(game, game.home)}")
    if game.need_home:
        lines.append("本關要求：灰塵清完後要回到充電座。")
    return "\n".join(lines)


print(describe_world(Game(2)))

### 3C. 記憶：禁區、已清掃區域、最近的指令

- **禁區**：玩家說「不准進臥室」會被規則式偵測寫入（也可在關卡預設）。
- **已清掃區域**、**最近幾句指令與結果**：每次組 Prompt 時帶入，讓行為跨指令保持一致。

In [ ]:
OP_TEXT = {"turn_left": "左轉", "turn_right": "右轉", "suck": "吸", "return_home": "回充", "stop": "停"}


def actions_to_text(actions):
    return "、".join(f"前進{a['n']}格" if a["op"] == "forward" else OP_TEXT[a["op"]] for a in actions) or "（沒動）"


def events_to_text(events):
    msgs = [e["msg"] for e in events if e["type"] not in ("navigating",)]
    return "；".join(msgs[:4]) if msgs else "順利完成，沒有意外。"


class Memory:
    def __init__(self, max_recent=4):
        self.recent = deque(maxlen=max_recent)

    def remember_turn(self, command, actions, events):
        self.recent.append({"cmd": command, "plan": actions_to_text(actions), "result": events_to_text(events)})

    def to_text(self, game):
        forb = "、".join(sorted(game.forbidden)) or "（無）"
        by_room = Counter(game.room_of(p) for p in game.cleaned)
        cleaned = "、".join(f"{r} {n} 坨" for r, n in by_room.items()) or "還沒有"
        lines = [f"- 禁區：{forb}（絕對不能進去）",
                 f"- 已清掃：{len(game.cleaned)}/{game.total_dirt} 坨（{cleaned}）"]
        if self.recent:
            lines.append("- 最近的指令：")
            for r in self.recent:
                lines.append(f"  ·「{r['cmd']}」→ 你做了：{r['plan']} → 結果：{r['result']}")
        return "\n".join(lines)


ROOM_ALIASES = {"房間": "臥室", "臥房": "臥室", "睡房": "臥室", "灶腳": "廚房"}
_ROOM_RE = "臥室|臥房|睡房|房間|廚房|灶腳|客廳"
FORBID_RE = re.compile(rf"(不准|不要|別|禁止|不可以|不能|不許)\s*再?\s*(進入|進去|跑進|跑去|靠近|進|去|到)?\s*(?P<room>{_ROOM_RE})")
ALLOW_RE = re.compile(rf"(?<!不)(可以|准|允許)\s*(進入|進去|進|去)\s*(?P<room>{_ROOM_RE})")


def update_rules_from_text(game, text):
    """從玩家的話更新禁區（寫入記憶）。回傳變更說明。"""
    notes = []
    for m in FORBID_RE.finditer(text):
        room = ROOM_ALIASES.get(m.group("room"), m.group("room"))
        if room in game.room_names() and room not in game.forbidden:
            game.forbidden.add(room)
            notes.append(f"📝 記住了：「{room}」是禁區")
    if not notes:
        for m in ALLOW_RE.finditer(text):
            room = ROOM_ALIASES.get(m.group("room"), m.group("room"))
            if room in game.forbidden:
                game.forbidden.discard(room)
                notes.append(f"📝 解除禁區：「{room}」")
    return notes


g3 = Game(3)
print(update_rules_from_text(g3, "不准進臥室！"), g3.forbidden)
print(update_rules_from_text(g3, "好啦現在可以進臥室了"), g3.forbidden)
print(Memory().to_text(g3))

### 3D. 輸出解析與白名單驗證

就算模型多講廢話、包了 \`\`\`json，也盡量把 JSON 挖出來；但**動作必須完全在選單內**，否則整份輸出視為無效。

In [ ]:
def extract_json(text):
    if not text or not text.strip():
        raise ValueError("空白回覆")
    t = text.strip()
    t = re.sub(r"^```[a-zA-Z]*\s*", "", t)
    t = re.sub(r"\s*```$", "", t)
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        pass
    s, e = t.find("{"), t.rfind("}")
    if s == -1 or e <= s:
        raise ValueError("找不到 JSON 物件")
    return json.loads(t[s:e + 1])


def validate_output(obj):
    """回傳 (plan, errors)。plan = {"actions": [...], "say": "..."}；只要有錯 plan 就是 None。"""
    if not isinstance(obj, dict):
        return None, ["最外層不是 JSON 物件"]
    acts = obj.get("actions")
    if not isinstance(acts, list):
        return None, ["缺少 actions 陣列"]
    errors, clean = [], []
    if len(acts) > MAX_ACTIONS:
        errors.append(f"動作太多（{len(acts)} > {MAX_ACTIONS}）")
    for i, a in enumerate(acts, 1):
        if not isinstance(a, dict) or "op" not in a:
            errors.append(f"第 {i} 個動作格式錯誤：{a!r}")
            continue
        op = a["op"]
        if op not in ACTION_WHITELIST:
            errors.append(f"不在選單內的動作：{op!r}")
            continue
        if op == "forward":
            n = a.get("n")
            if isinstance(n, str) and n.strip().isdigit():
                n = int(n)
            if isinstance(n, float) and n.is_integer():
                n = int(n)
            if not isinstance(n, int) or isinstance(n, bool) or not 1 <= n <= MAX_FORWARD:
                errors.append(f"forward 的 n 不合法：{a.get('n')!r}")
                continue
            clean.append({"op": "forward", "n": n})
        else:
            clean.append({"op": op})
    if errors:
        return None, errors
    say = obj.get("say", "")
    say = say if isinstance(say, str) else str(say)
    return {"actions": clean, "say": say.strip()[:120]}, []


# 小測試
tests = [
    '{"actions": [{"op": "forward", "n": 3}, {"op": "turn_left"}], "say": "好"}',
    '```json\n{"actions": [{"op": "suck"}], "say": "吸！"}\n```',
    '我覺得應該 {"actions": [{"op": "dance"}], "say": "跳舞"}',
    '{"actions": [{"op": "forward", "n": 3},], "say": "多一個逗號"}',
]
for t in tests:
    try:
        print(validate_output(extract_json(t)))
    except (ValueError, json.JSONDecodeError) as exc:
        print("解析失敗：", exc)

### 3E. 呼叫 LLM（含離線 FakeLLM）

- `USE_JSON_MODE = True` 會要求 vLLM 強制輸出 JSON（`response_format`），格式正確率通常會提高；
  預設關閉，才能真實量到模型「自己」的格式正確率。
- `FakeLLM` 是規則式假模型，只看得懂簡單中文，賣蠢模式下會隨機左右顛倒、多走十倍、或吐出壞掉的 JSON。

In [ ]:
USE_JSON_MODE = False
_json_mode_supported = True

CN_DIGITS = {"零": 0, "一": 1, "二": 2, "兩": 2, "三": 3, "四": 4, "五": 5, "六": 6, "七": 7, "八": 8, "九": 9}
NUM_RE = r"([0-9]+|[一二兩三四五六七八九十]+)"


def cn_to_int(s):
    if s.isdigit():
        return int(s)
    if "十" in s:
        a, _, b = s.partition("十")
        return CN_DIGITS.get(a, 1) * 10 + CN_DIGITS.get(b, 0)
    return CN_DIGITS.get(s, 1)


def parse_simple_command(cmd):
    """FakeLLM 用的超簡易中文指令解析（真正的理解交給 LLM）。"""
    acts = []
    for c in re.split(r"[，,。、；;！!]|然後|接著|之後|再", cmd):
        c = c.strip()
        if not c:
            continue
        if re.search(r"停|不要動|別動", c):
            acts.append({"op": "stop"})
        times = re.search(NUM_RE + r"\s*次", c)
        k = cn_to_int(times.group(1)) if times else 1
        if "吸" in c and not re.search(r"走|前進", c):
            acts += [{"op": "suck"}] * k
        if re.search(r"回充|充電|回家|回去", c):
            acts.append({"op": "return_home"})
            continue
        if re.search(r"向後轉|掉頭|迴轉|轉身|往後|向後|後退", c):
            acts += [{"op": "turn_right"}] * 2
        elif "左" in c:
            acts += [{"op": "turn_left"}] * (k if "轉" in c else 1)
        elif "右" in c:
            acts += [{"op": "turn_right"}] * (k if "轉" in c else 1)
        n = re.search(NUM_RE + r"\s*(格|步)", c)
        if n or re.search(r"前進|往前|走", c):
            acts.append({"op": "forward", "n": cn_to_int(n.group(1)) if n else 1})
    return acts


class FakeLLM:
    """離線測試用：看得懂簡單中文指令，賣蠢模式下會隨機出錯。"""
    SILLY = ["收到！不過我覺得那隻貓比地板還髒。", "好啦好啦，你說了算。", "先聲明，撞到不是我的錯。",
             "這個方向我有不同看法，但還是照做。", "掃地是我的天命！", "我覺得那邊比較髒。"]
    SERIOUS = ["收到，開始執行。", "了解。", "好的。"]

    def __init__(self, seed=42):
        self.rng = random.Random(seed)

    def chat(self, messages, temperature=0.5):
        time.sleep(self.rng.uniform(0.3, 0.9))     # 假裝在思考
        silly = "笨笨" in messages[0]["content"]
        last = messages[-1]["content"]
        if "【執行結果】" in last:
            say = self.rng.choice(["哎呀，這不在我的計畫內……", "是那個東西先擋我的！", "沒事沒事，小意外。"])
            return json.dumps({"actions": [], "say": say}, ensure_ascii=False)
        first = next(m["content"] for m in messages if m["role"] == "user")
        m = re.search(r"【主人指令】\s*(.+)", first, re.S)
        actions = parse_simple_command(m.group(1).strip() if m else first)
        say = self.rng.choice(self.SILLY if silly else self.SERIOUS)
        if silly and not last.startswith("你的回覆不符合格式"):
            r = self.rng.random()
            if r < 0.10:
                return '嗯……我覺得應該先 {"actions": [{"op": "dance"}], "say": "跳支舞'   # 壞掉的 JSON
            if r < 0.25:   # 左右搞反
                swap = {"turn_left": "turn_right", "turn_right": "turn_left"}
                actions = [dict(a, op=swap.get(a["op"], a["op"])) for a in actions]
                say = "左右是相對的嘛～"
            elif r < 0.32:  # 三格走成三十格
                actions = [dict(a, n=min(a["n"] * 10, MAX_FORWARD)) if a["op"] == "forward" else a for a in actions]
                say = "多走一點比較乾淨！"
        return json.dumps({"actions": actions, "say": say}, ensure_ascii=False)


FAKE_LLM = FakeLLM()


def chat_llm(messages, temperature=0.5, max_tokens=300):
    """送出 messages，回傳模型的文字回覆。"""
    global _json_mode_supported
    if USE_FAKE_LLM:
        return FAKE_LLM.chat(messages, temperature)
    kwargs = dict(model=MODEL, messages=messages, temperature=temperature, max_tokens=max_tokens, stream=False)
    if USE_JSON_MODE and _json_mode_supported:
        try:
            resp = client.chat.completions.create(response_format={"type": "json_object"}, **kwargs)
            return resp.choices[0].message.content or ""
        except Exception as exc:
            if getattr(exc, "status_code", None) not in (400, 422):
                raise
            _json_mode_supported = False
            print("⚠️ 伺服器不支援 response_format，改回一般模式。")
    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content or ""


print(parse_simple_command("往右走兩格然後吸三次，再回去充電"))

### 3F. Agent：組 Prompt → 呼叫 LLM → 解析驗證 →（失敗重試一次）→ 發呆演出

這就是提案中的「指令理解 → 規劃動作」那一段。`think()` 只回傳**結構化結果**，不直接動到遊戲世界。

In [ ]:
DAZE_ACTIONS = [
    [],                                                                       # 原地發呆
    [{"op": "turn_left"}] * 4,                                                # 原地轉一圈
    [{"op": "turn_right"}, {"op": "turn_left"}, {"op": "turn_right"}, {"op": "turn_left"}],  # 原地亂扭
]


class Agent:
    def __init__(self, mode="silly", silliness=None):
        self.mode = mode              # "silly"＝賣蠢模式；"serious"＝正經模式（對照組）
        self.silliness = silliness    # None＝依關卡設定
        self.memory = Memory()

    def system_prompt(self, game):
        if self.mode == "serious":
            persona = PERSONA_SERIOUS
        else:
            s = game.cfg.get("silliness", 1) if self.silliness is None else self.silliness
            persona = PERSONA_SILLY + SILLY_STRENGTH[s]
        return persona + "\n\n" + ACTION_SPEC

    def build_messages(self, game, command, feedback_events=None):
        world, mem = describe_world(game), self.memory.to_text(game)
        if feedback_events is None:
            user = f"【世界狀態】\n{world}\n\n【記憶】\n{mem}\n\n【主人指令】\n{command}"
        else:
            user = (f"【執行結果】你剛剛為了主人的指令「{command}」執行動作，結果：{events_to_text(feedback_events)}\n\n"
                    f"【世界狀態】\n{world}\n\n【記憶】\n{mem}\n\n"
                    "請用 say 對主人說一句話回應這個結果。如果需要修正、完成原本的指令，可以在 actions 補上動作；"
                    "不需要就給空陣列 []。同樣只輸出 JSON。")
        return [{"role": "system", "content": self.system_prompt(game)}, {"role": "user", "content": user}]

    def think(self, game, command, feedback_events=None, lock=None):
        with (lock or nullcontext()):          # 讀取世界狀態時上鎖（背景執行緒安全）
            messages = self.build_messages(game, command, feedback_events)
        temperature = 0.2 if self.mode == "serious" else 0.8
        result = {"command": command, "mode": self.mode, "raws": [], "errors": [],
                  "format_ok": False, "retried": False, "ok": False, "dazed": False,
                  "actions": [], "say": ""}
        t0 = time.perf_counter()
        for attempt in range(2):                # 解析失敗自動重試一次
            try:
                raw = chat_llm(messages, temperature=temperature)
            except Exception as exc:
                result["errors"].append(f"API 錯誤：{type(exc).__name__}: {str(exc)[:100]}")
                break
            result["raws"].append(raw)
            try:
                plan, errs = validate_output(extract_json(raw))
            except (ValueError, json.JSONDecodeError) as exc:
                plan, errs = None, [f"JSON 解析失敗：{exc}"]
            if plan is not None:
                result.update(plan, ok=True, format_ok=(attempt == 0))
                break
            result["errors"] += errs
            if attempt == 0:
                result["retried"] = True
                messages = messages + [
                    {"role": "assistant", "content": raw},
                    {"role": "user", "content": "你的回覆不符合格式：" + "；".join(errs[:3])
                     + '。請只輸出一個 JSON 物件，例如 {"actions": [{"op": "forward", "n": 1}], "say": "好"}'},
                ]
        result["latency"] = time.perf_counter() - t0
        if not result["ok"]:                    # 兩次都失敗 → 發呆演出，把模型原話秀出來
            raw0 = result["raws"][-1].strip() if result["raws"] else ""
            result["actions"] = random.choice(DAZE_ACTIONS)
            result["say"] = "（發呆中……）" + (f"我剛剛想說：「{raw0[:60]}」" if raw0 else "腦袋連不上線。")
            result["dazed"] = True
        return result

### 3G. 試一句指令：看看 LLM 的原話、驗證後的動作、執行結果

In [ ]:
g = Game(1)
agent = Agent("silly")
cmd = "往前走一格，然後左轉走兩格，把灰塵吸掉"

print("===== 送給模型的 user 訊息 =====")
print(agent.build_messages(g, cmd)[1]["content"])

res = agent.think(g, cmd)
print("\n===== 模型原話 =====")
for raw in res["raws"]:
    print(raw)
print("\n格式第一次就正確？", res["format_ok"], "｜有重試？", res["retried"], f"｜延遲 {res['latency']:.2f}s")
print("驗證後動作：", actions_to_text(res["actions"]))
print("🤖 說：", res["say"])

g.set_plan(res["actions"])
for e in run_to_end(g):
    print(f"[{e['type']}] {e['msg']}")
show(g)

## 4｜🔁 Session：完整 Agent 迴圈＋非同步＋事件回饋＋Log

```
玩家指令 ─► preprocess（「停」不經 LLM、禁區寫入記憶）
         ─► think（背景執行緒呼叫 LLM，畫面不卡住）
         ─► apply（動作放進佇列，取代舊計畫）
         ─► tick（每次走一步、收集事件）
         ─► 計畫結束：寫 log、寫記憶；若被中斷 → 把事件回饋給 LLM 再回應一次（閉環）
```
每一回合寫成一行 JSON 到 `logs/game_log.jsonl`，第 6 節會自動算評估指標。

In [ ]:
class JsonlLogger:
    def __init__(self, path=LOG_DIR / "game_log.jsonl"):
        self.path = Path(path)
        self._lock = threading.Lock()

    def write(self, record):
        with self._lock, open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


STOP_RE = re.compile(r"\s*(停|停止|停下|停下來|煞車|別動|stop)[!！。.\s]*", re.I)
EXECUTOR = ThreadPoolExecutor(max_workers=2)


class Session:
    def __init__(self, level_id=1, mode="silly", feedback_rounds=1, logger=None):
        self.lock = threading.RLock()
        self.logger = logger or JsonlLogger()
        self.agent = Agent(mode)
        self.feedback_rounds = feedback_rounds   # 事件回饋最多幾輪（0 = 不回饋）
        self.reset(level_id)

    def reset(self, level_id=None):
        with self.lock:
            self.game = Game(level_id if level_id is not None else self.game.level_id)
            self.agent.memory = Memory()
            self.id = uuid.uuid4().hex[:8]
            self.current = None
            self.pending_feedback = None
            self.n_commands = 0

    # ---- 1) 玩家輸入的本地處理 ----
    def preprocess(self, command):
        with self.lock:
            self.n_commands += 1
            notes = update_rules_from_text(self.game, command)
        if STOP_RE.fullmatch(command):            # 保底動作：「停」直接處理，不等 LLM
            self.stop(command)
            return {"handled": True, "say": "收到！緊急煞車！", "notes": notes}
        return {"handled": False, "notes": notes}

    def stop(self, command="停"):
        with self.lock:
            self.game.queue.clear()
            ev = E("stopped", "收到「停」，動作佇列已清空。")
            if self.current:
                self.current["events"].append(ev)
                self._finish()
            self.pending_feedback = None
            self._log({"kind": "local_stop", "command": command})
        return [ev]

    def local_plan(self, actions, label):
        """保底按鈕（例如「回充」）：不經 LLM 直接執行。"""
        with self.lock:
            self.n_commands += 1
        res = {"actions": actions, "say": label, "ok": True, "format_ok": True, "raws": [], "latency": 0.0}
        self.apply(label, res, kind="local")

    # ---- 2) 思考（在背景執行緒呼叫）----
    def think(self, command, feedback=None):
        events = feedback["events"] if feedback else None
        return self.agent.think(self.game, command, feedback_events=events, lock=self.lock)

    # ---- 3) 套用計畫 ----
    def apply(self, command, result, kind="command", depth=0):
        with self.lock:
            if self.current:
                self._finish(replaced=True)
            self.game.set_plan(result["actions"])
            self.current = {"command": command, "kind": kind, "depth": depth, "result": result, "events": []}
            if not self.game.queue:
                self._finish()

    # ---- 4) 每個 tick 執行一步 ----
    def tick(self):
        with self.lock:
            events = self.game.step()
            if self.current is not None:
                self.current["events"] += events
                if not self.game.queue:
                    self._finish()
            return events

    def busy(self):
        return bool(self.game.queue)

    def pop_feedback(self):
        with self.lock:
            fb, self.pending_feedback = self.pending_feedback, None
            return fb

    # ---- 5) 計畫結束：log＋記憶＋決定要不要回饋 ----
    def _finish(self, replaced=False):
        cur, self.current = self.current, None
        r, evs = cur["result"], cur["events"]
        types = [e["type"] for e in evs]
        interrupted = any(t in INTERRUPT_EVENTS for t in types)
        self._log({
            "kind": cur["kind"], "depth": cur["depth"], "command": cur["command"],
            "raw_outputs": r.get("raws", []), "format_ok": r.get("format_ok"), "retried": r.get("retried", False),
            "parse_ok": r.get("ok"), "dazed": r.get("dazed", False), "errors": r.get("errors", [])[:5],
            "actions": r.get("actions", []), "say": r.get("say", ""), "latency_s": round(r.get("latency", 0.0), 3),
            "events": types, "interrupted": interrupted, "replaced": replaced,
            "forbidden_violation": "entered_forbidden" in types,
        })
        label = cur["command"] if cur["kind"] != "feedback" else f"（修正）{cur['command']}"
        self.agent.memory.remember_turn(label, r.get("actions", []), evs)
        if (not replaced and interrupted and cur["kind"] != "local"
                and cur["depth"] < self.feedback_rounds and not self.game.cleared):
            self.pending_feedback = {"command": cur["command"], "events": evs, "depth": cur["depth"] + 1}

    def _log(self, rec):
        g = self.game
        base = {"ts": datetime.now().isoformat(timespec="seconds"), "session": self.id, "level": g.level_id,
                "mode": self.agent.mode, "model": current_model_name(), "n_commands": self.n_commands,
                "level_cleared": g.cleared, "dirt_left": len(g.dirt), "forbidden_rooms": sorted(g.forbidden),
                "violations_total": g.violations}
        self.logger.write({**base, **rec})


print("✅ Session 準備完成，log 會寫到", (LOG_DIR / "game_log.jsonl").resolve())

## 5｜🎮 開始玩！

### 5A. 對話框與畫面更新（共用）

In [ ]:
EVENT_ICON = {"collision": "💥", "lego": "🧱", "tangled": "🔌", "jammed": "🧦", "sucked_cat": "🐱",
              "entered_forbidden": "🚫", "cleaned": "✨", "level_clear": "🎉", "docked": "🔋",
              "navigating": "🧭", "stopped": "✋", "suck_empty": "💨", "no_path": "❓", "rule": "📝"}


def hud_html(session, say="", events=(), thinking=False):
    g = session.game
    mode = "賣蠢模式" if session.agent.mode == "silly" else "正經模式"
    bubble = "🤔 思考中……（機器人可能還在執行上一串動作）" if thinking else html.escape(say or "……")
    items = "".join(f"<li>{EVENT_ICON.get(e['type'], '•')} {html.escape(e['msg'])}</li>" for e in list(events)[-5:])
    forb = "、".join(sorted(g.forbidden)) or "無"
    clear = "<div style='color:#2a9d8f;font-weight:bold;font-size:16px'>🎉 過關！可以換下一關了</div>" if g.cleared else ""
    return (
        "<div style='font-family:sans-serif;max-width:640px;color:#222'>"
        "<div style='background:#fff8e1;border:2px solid #f4a261;border-radius:12px;padding:8px 12px;margin:4px 0'>"
        f"<b>🤖 笨笨：</b>{bubble}</div>"
        f"<div style='font-size:13px;color:#555'>第 {g.level_id} 關「{html.escape(g.cfg['name'])}」｜{mode}｜"
        f"灰塵 {len(g.dirt)}/{g.total_dirt}｜禁區：{html.escape(forb)}｜指令數 {session.n_commands}｜違規 {g.violations}</div>"
        f"<ul style='font-size:13px;margin:4px 0;padding-left:20px'>{items}</ul>{clear}</div>"
    )


class NotebookView:
    """文字模式用：在同一個位置更新畫面與對話框。"""
    def __init__(self, session):
        self.session = session
        self.say = session.game.cfg["intro"]
        self.events = deque(maxlen=5)
        self.frame = 0
        self.h_img = display(HTML("<i>載入畫面中……</i>"), display_id=True)
        self.h_txt = display(HTML(""), display_id=True)
        self.update()

    def update(self, say=None, events=(), thinking=False):
        if say is not None:
            self.say = say
        self.events.extend(events)
        self.frame += 1
        with self.session.lock:
            png = render_png(self.session.game, thinking=thinking, frame=self.frame)
            txt = hud_html(self.session, self.say, self.events, thinking)
        self.h_img.update(Image(data=png))
        self.h_txt.update(HTML(txt))

### 5B. 文字模式（最穩定，任何 Jupyter 都能玩）

執行下一格後，在輸入框打指令。特殊指令：`/level 2` 換關、`/mode serious` 或 `/mode silly` 換模式、`/reset` 重來、`q` 離開。

試試看：`往前走三格然後吸`、`右轉，前進兩格`、`不准進臥室`、`繞過那隻貓`、`回去充電`、`停`

In [ ]:
def _wait_thinking(session, view, fut, tick):
    """LLM 在背景思考時：畫面持續更新「思考中」，機器人繼續執行還沒做完的動作。"""
    while not fut.done():
        view.update(events=session.tick(), thinking=True)
        time.sleep(tick)
    return fut.result()


def _run_until_idle(session, view, tick):
    while session.busy():
        view.update(events=session.tick())
        time.sleep(tick)
    view.update()


def run_turn(session, view, command, tick=0.25):
    pre = session.preprocess(command)
    notes = [E("rule", n) for n in pre["notes"]]
    if pre["handled"]:
        view.update(say=pre["say"], events=notes)
        return
    view.update(events=notes, thinking=True)
    res = _wait_thinking(session, view, EXECUTOR.submit(session.think, command), tick)
    session.apply(command, res)
    view.update(say=res["say"] or "……")
    _run_until_idle(session, view, tick)
    # 事件回饋：被中斷時，把事件交回 LLM 回應／修正（Agent 閉環）
    while (fb := session.pop_feedback()):
        res = _wait_thinking(session, view, EXECUTOR.submit(session.think, fb["command"], fb), tick)
        session.apply(fb["command"], res, kind="feedback", depth=fb["depth"])
        view.update(say=res["say"] or "……")
        _run_until_idle(session, view, tick)


def play(level=1, mode="silly", tick=0.25, feedback_rounds=1):
    session = Session(level, mode, feedback_rounds)
    view = NotebookView(session)
    print("指令範例：往前走三格然後吸｜不准進臥室｜回去充電｜停　（/level N、/mode serious|silly、/reset、q 離開）")
    while True:
        try:
            cmd = input("🧹 你的指令 > ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not cmd:
            continue
        if cmd.lower() in ("q", "quit", "exit", "離開"):
            break
        if cmd.startswith("/level"):
            try:
                session.reset(int(cmd.split()[1]))
                view.update(say=session.game.cfg["intro"])
            except (IndexError, ValueError, KeyError):
                print("用法：/level 1、/level 2、/level 3")
            continue
        if cmd.startswith("/mode"):
            session.agent.mode = "serious" if "serious" in cmd else "silly"
            view.update(say=f"切換成{'正經' if session.agent.mode == 'serious' else '賣蠢'}模式。")
            continue
        if cmd == "/reset":
            session.reset()
            view.update(say=session.game.cfg["intro"])
            continue
        run_turn(session, view, cmd, tick)
    print("👋 遊戲結束，log 存在：", session.logger.path.resolve())
    return session

In [ ]:
# ▶️ 執行這格開始玩（文字模式）
session = play(level=1, mode="silly")

### 5C. 即時模式（ipywidgets，選用）

更接近「即時制」：遊戲迴圈在背景執行緒持續跑，LLM 也在背景思考。
**機器人執行動作時你可以隨時按「停！」**——趕在它撞上樂高之前！
（需要 `ipywidgets`；若畫面沒有更新，請改用上面的文字模式。）

In [ ]:
_ACTIVE_UI = None


class WidgetGame:
    def __init__(self, level=1, mode="silly", tick=0.35, feedback_rounds=1):
        import ipywidgets as W
        self.session = Session(level, mode, feedback_rounds)
        self.tick_s = tick
        self.say = self.session.game.cfg["intro"]
        self.events = deque(maxlen=5)
        self.future, self.future_info = None, None
        self.dirty, self.frame = True, 0

        self.img = W.Image(format="png", layout=W.Layout(width="640px"))
        self.hud = W.HTML()
        self.cmd = W.Text(placeholder="輸入指令後按 Enter，例如：往前走三格然後吸",
                          continuous_update=False, layout=W.Layout(width="360px"))
        self.btn_send = W.Button(description="送出", button_style="primary")
        self.btn_stop = W.Button(description="停！", button_style="danger")
        self.btn_home = W.Button(description="回充")
        levels = [(f"第 {k} 關：{v['name']}", k) for k, v in LEVELS.items() if isinstance(k, int)]
        self.dd_level = W.Dropdown(options=levels, value=level, layout=W.Layout(width="210px"))
        self.tb_mode = W.ToggleButtons(options=[("賣蠢模式", "silly"), ("正經模式", "serious")], value=mode)
        self.btn_reset = W.Button(description="重來")
        self.btn_quit = W.Button(description="結束遊戲")

        self.cmd.observe(self._on_cmd, names="value")
        self.btn_send.on_click(lambda _: self._take_input())
        self.btn_stop.on_click(self._on_stop)
        self.btn_home.on_click(self._on_home)
        self.dd_level.observe(lambda ch: self._reset(ch["new"]), names="value")
        self.tb_mode.observe(self._on_mode, names="value")
        self.btn_reset.on_click(lambda _: self._reset(self.session.game.level_id))
        self.btn_quit.on_click(lambda _: self.quit())

        self.box = W.VBox([
            W.HBox([self.dd_level, self.tb_mode, self.btn_reset, self.btn_quit]),
            self.img, self.hud,
            W.HBox([self.cmd, self.btn_send, self.btn_stop, self.btn_home]),
        ])
        self.alive = threading.Event()
        self.alive.set()
        self._render(force=True)
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    # ---------- 玩家操作 ----------
    def _on_cmd(self, change):
        if change["new"].strip():
            self._take_input()

    def _take_input(self):
        text = self.cmd.value.strip()
        self.cmd.value = ""
        if not text:
            return
        if self.future is not None:
            self.say = "等等，我還在想上一句！（急的話先按「停！」）"
            self.dirty = True
            return
        pre = self.session.preprocess(text)
        self.events.extend(E("rule", n) for n in pre["notes"])
        if pre["handled"]:
            self.say = pre["say"]
        else:
            self.future = EXECUTOR.submit(self.session.think, text)
            self.future_info = {"command": text, "kind": "command", "depth": 0}
        self.dirty = True

    def _on_stop(self, _):
        self.events.extend(self.session.stop())
        if self.future_info:
            self.future_info["discard"] = True    # 正在想的那句也不要了
        self.say = "收到！緊急煞車！"
        self.dirty = True

    def _on_home(self, _):
        self.session.local_plan([{"op": "return_home"}], "回充")
        self.say = "好啦，回家充電。"
        self.dirty = True

    def _on_mode(self, change):
        self.session.agent.mode = change["new"]
        self.say = f"切換成{'正經' if change['new'] == 'serious' else '賣蠢'}模式。"
        self.dirty = True

    def _reset(self, level):
        if self.future_info:
            self.future_info["discard"] = True
        self.session.reset(level)
        self.events.clear()
        self.say = self.session.game.cfg["intro"]
        self.dirty = True

    # ---------- 背景遊戲迴圈 ----------
    def _loop(self):
        while self.alive.is_set():
            try:
                ev = self.session.tick()
                if ev:
                    self.events.extend(ev)
                    self.dirty = True
                if self.future is not None and self.future.done():
                    fut, info = self.future, self.future_info
                    self.future = self.future_info = None
                    try:
                        res = fut.result()
                    except Exception as exc:
                        res, self.say = None, f"（大腦當機：{type(exc).__name__}）"
                    if res is not None and not info.get("discard"):
                        self.session.apply(info["command"], res, kind=info["kind"], depth=info["depth"])
                        self.say = res["say"] or "……"
                    self.dirty = True
                if self.future is None:
                    fb = self.session.pop_feedback()
                    if fb:
                        self.future = EXECUTOR.submit(self.session.think, fb["command"], fb)
                        self.future_info = {"command": fb["command"], "kind": "feedback", "depth": fb["depth"]}
                        self.dirty = True
                self._render()
            except Exception as exc:
                self.hud.value = f"<pre>背景迴圈錯誤：{html.escape(repr(exc))}</pre>"
            time.sleep(self.tick_s)

    def _render(self, force=False):
        thinking = self.future is not None
        if not (force or self.dirty or thinking):
            return
        self.dirty = False
        self.frame += 1
        with self.session.lock:
            png = render_png(self.session.game, thinking=thinking, frame=self.frame)
            txt = hud_html(self.session, self.say, self.events, thinking)
        self.img.value = png
        self.hud.value = txt

    def quit(self):
        self.alive.clear()
        for w in (self.cmd, self.btn_send, self.btn_stop, self.btn_home, self.dd_level,
                  self.tb_mode, self.btn_reset, self.btn_quit):
            w.disabled = True
        self.say = "遊戲結束，謝謝遊玩！"
        self._render(force=True)


def play_widgets(level=1, mode="silly", tick=0.35, feedback_rounds=1):
    global _ACTIVE_UI
    try:
        import ipywidgets  # noqa: F401
    except ImportError:
        print("沒有 ipywidgets：請改用 play() 文字模式，或執行 %pip install ipywidgets")
        return None
    if _ACTIVE_UI is not None:
        _ACTIVE_UI.quit()
    _ACTIVE_UI = WidgetGame(level, mode, tick, feedback_rounds)
    display(_ACTIVE_UI.box)
    return _ACTIVE_UI

In [ ]:
# ▶️ 執行這格開始玩（即時模式）
ui = play_widgets(level=1, mode="silly")

## 6｜📊 評估：LLM 作為 Agent 的表現

| 指標 | 定義 | 資料來源 |
|---|---|---|
| **照做率** | 30 句固定測試指令，比對 LLM 動作序列與人工標註是否一致；正經／賣蠢模式分開統計 | 6A 測試集 |
| **格式正確率** | 第一次回覆就能解析、且動作皆在選單內的比例 | 自動計算 |
| **記憶遵守率** | 已設定禁區後，模型規劃的動作「沒有」闖進禁區的比例 | 6A 記憶測試＋遊戲 log |
| **回應延遲** | 送出指令到拿到動作的秒數 | 自動計算 |
| **通關所需指令數** | 每關從開始到通關平均下了幾句指令（越少越聽話） | 遊戲 log |

比對時會先「正規化」：連續前進合併（前進 2 + 前進 1 = 前進 3），連續轉向換算成淨角度（左轉兩次 = 右轉兩次 = 向後轉）。

### 6A. 測試指令集（人工標註）

測試場景固定：機器人在客廳、面向東；**前方 2 格有灰塵、前方 4 格有貓**；左手邊（北方）2 格外是臥室。
標註寫法：`F3`＝前進 3 格、`L`／`R`＝左／右轉、`S`＝吸、`H`＝回充、`X`＝停；多種正確答案用 `|` 分開。

In [ ]:
LEVELS["eval"] = {
    "name": "評估場景",
    "intro": "評估用固定場景",
    "map": [
        "#########",
        "#...D...#",
        "#.......#",
        "#.......#",
        "#.>.D.C.#",
        "#.......#",
        "#H......#",
        "#########",
    ],
    "rooms": {"臥室": (1, 1, 7, 2)},
    "forbidden": [],
    "need_home": False,
    "silliness": 2,
}

TEST_SET = [
    {"id": 1, "cmd": "前進 3 格", "expect": ["F3"]},
    {"id": 2, "cmd": "往前走兩格", "expect": ["F2"]},
    {"id": 3, "cmd": "左轉", "expect": ["L"]},
    {"id": 4, "cmd": "右轉", "expect": ["R"]},
    {"id": 5, "cmd": "向後轉", "expect": ["RR"]},
    {"id": 6, "cmd": "往右走 2 格", "expect": ["R F2"]},
    {"id": 7, "cmd": "往左走一格", "expect": ["L F1"]},
    {"id": 8, "cmd": "前進一格然後右轉", "expect": ["F1 R"]},
    {"id": 9, "cmd": "右轉、前進 3 格、再左轉", "expect": ["R F3 L"]},
    {"id": 10, "cmd": "吸一下", "expect": ["S"]},
    {"id": 11, "cmd": "停", "expect": ["X", ""]},
    {"id": 12, "cmd": "回去充電", "expect": ["H"]},
    {"id": 13, "cmd": "前進 2 格然後吸地", "expect": ["F2 S"]},
    {"id": 14, "cmd": "去把前面的灰塵吸掉", "expect": ["F2 S"]},
    {"id": 15, "cmd": "往後走 2 格", "expect": ["RR F2"]},
    {"id": 16, "cmd": "左轉兩次", "expect": ["RR"]},
    {"id": 17, "cmd": "先右轉再右轉再前進 1 格", "expect": ["RR F1"]},
    {"id": 18, "cmd": "前進五格", "expect": ["F5"]},
    {"id": 19, "cmd": "往前一步", "expect": ["F1"]},
    {"id": 20, "cmd": "吸完回充電座", "expect": ["S H"]},
    {"id": 21, "cmd": "往右轉然後前進四格再吸", "expect": ["R F4 S"]},
    {"id": 22, "cmd": "掉頭前進 3 格", "expect": ["RR F3"]},
    {"id": 23, "cmd": "前進 1 格、左轉、前進 1 格、右轉", "expect": ["F1 L F1 R"]},
    {"id": 24, "cmd": "不要動", "expect": ["", "X"]},
    {"id": 25, "cmd": "走到灰塵那裡", "expect": ["F2"]},
    {"id": 26, "cmd": "左轉後走 3 格然後回充", "expect": ["L F3 H"]},
    {"id": 27, "cmd": "右轉 3 次", "expect": ["L"]},
    {"id": 28, "cmd": "前進 2 格，再前進 1 格", "expect": ["F3"]},
    {"id": 29, "cmd": "原地吸三次", "expect": ["S S S"]},
    {"id": 30, "cmd": "停止所有動作然後回家", "expect": ["X H", "H"]},
]

# 記憶遵守測試：先把「臥室」設為禁區，再下會誘惑它進臥室的指令
MEMORY_TESTS = [
    "往左走 3 格",
    "左轉然後前進 2 格",
    "去吸臥室裡的灰塵",
    "往北走兩格",
    "左轉，一直往前走到底",
    "去左邊那坨灰塵那裡吸一下",
]

TOKEN_OP = {"L": "turn_left", "R": "turn_right", "S": "suck", "H": "return_home", "X": "stop"}


def short_to_actions(s):
    acts = []
    for tok in s.split():
        if tok[0] == "F":
            acts.append({"op": "forward", "n": int(tok[1:])})
        else:
            acts += [{"op": TOKEN_OP[c]} for c in tok]
    return acts


def normalize(actions):
    """動作序列正規化：合併連續前進、轉向換算成淨角度。"""
    out, rot, fwd = [], 0, 0

    def flush_rot():
        nonlocal rot
        out.extend({0: [], 1: ["R"], 2: ["R", "R"], 3: ["L"]}[rot % 4])
        rot = 0

    def flush_fwd():
        nonlocal fwd
        if fwd:
            out.append(f"F{fwd}")
        fwd = 0

    for a in actions:
        op = a["op"]
        if op in ("turn_left", "turn_right"):
            flush_fwd()
            rot += 1 if op == "turn_right" else -1
        elif op == "forward":
            flush_rot()
            fwd += a["n"]
        else:
            flush_rot()
            flush_fwd()
            out.append({"suck": "S", "return_home": "H", "stop": "X"}[op])
    flush_rot()
    flush_fwd()
    return " ".join(out)


assert normalize(short_to_actions("LL F2")) == normalize(short_to_actions("RR F2")) == "R R F2"
print(f"測試指令 {len(TEST_SET)} 句、記憶測試 {len(MEMORY_TESTS)} 句")
print(describe_world(Game("eval")))

### 6B. 跑評估（正經模式 vs 賣蠢模式）

`models` 可以放多個模型一起比較，例如 `["qwen2.5:3b", "qwen2.5:7b", "llama3.1:8b"]`；`repeat` 可重複多次減少隨機性。
每句都會真的呼叫 API，請留意時間（約 `句數 × 模式數 × 模型數 × 延遲`）。

In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None
    print("⚠️ 沒有 pandas：請執行 %pip install pandas")


def run_eval(modes=("serious", "silly"), models=None, repeat=1, verbose=True):
    global MODEL
    original, rows = MODEL, []
    try:
        for model in (models or [MODEL]):
            MODEL = model
            for mode in modes:
                for rep in range(repeat):
                    for t in TEST_SET:
                        game, agent = Game("eval"), Agent(mode)
                        res = agent.think(game, t["cmd"])
                        got = normalize(res["actions"]) if res["ok"] else None
                        expect = [normalize(short_to_actions(e)) for e in t["expect"]]
                        rows.append({"model": current_model_name(), "mode": mode, "rep": rep, "type": "instruction",
                                     "id": t["id"], "cmd": t["cmd"], "format_ok": res["format_ok"],
                                     "parse_ok": res["ok"], "follow": got in expect, "got": got,
                                     "expect": " | ".join(expect), "latency_s": res["latency"], "violated": None,
                                     "raw": res["raws"][-1] if res["raws"] else ""})
                    for i, cmd in enumerate(MEMORY_TESTS, 1):
                        game, agent = Game("eval"), Agent(mode)
                        game.forbidden = {"臥室"}
                        res = agent.think(game, cmd)
                        sim, _ = simulate(game, res["actions"])
                        rows.append({"model": current_model_name(), "mode": mode, "rep": rep, "type": "memory",
                                     "id": f"M{i}", "cmd": cmd, "format_ok": res["format_ok"], "parse_ok": res["ok"],
                                     "follow": None, "got": normalize(res["actions"]), "expect": "不進臥室",
                                     "latency_s": res["latency"], "violated": sim.violations > 0,
                                     "raw": res["raws"][-1] if res["raws"] else ""})
                    if verbose:
                        print(f"✔ {current_model_name()}｜{mode}｜第 {rep + 1} 輪完成")
    finally:
        MODEL = original
    df = pd.DataFrame(rows)
    out = LOG_DIR / f"eval_{datetime.now():%m%d_%H%M%S}.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")
    print("📁 已存檔：", out)
    return df


def summarize_eval(df):
    ins = df[df["type"] == "instruction"].groupby(["model", "mode"]).agg(
        格式正確率=("format_ok", "mean"), 可解析率=("parse_ok", "mean"),
        照做率=("follow", "mean"), 平均延遲秒=("latency_s", "mean"))
    mem = df[df["type"] == "memory"].groupby(["model", "mode"]).agg(
        記憶遵守率=("violated", lambda s: 1 - s.astype(float).mean()))
    return ins.join(mem).round(3)


eval_df = run_eval(modes=("serious", "silly"))
summary = summarize_eval(eval_df)
summary

### 6C. 畫圖比較 & 看看哪幾句最容易「賣蠢」

In [ ]:
def plot_summary(summary):
    metrics = [("格式正確率", "format OK"), ("照做率", "follow rate"), ("記憶遵守率", "memory compliance")]
    fig = Figure(figsize=(8, 4), dpi=90)
    FigureCanvasAgg(fig)
    ax = fig.add_subplot()
    labels = [f"{m}\n{mode}" for m, mode in summary.index]
    width = 0.8 / len(metrics)
    for i, (col, en) in enumerate(metrics):
        xs = [k + i * width for k in range(len(summary))]
        bars = ax.bar(xs, summary[col], width, label=L(col, en))
        for b, v in zip(bars, summary[col]):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.0%}", ha="center", fontsize=8)
    ax.set_xticks([k + width for k in range(len(summary))])
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("rate")
    ax.set_title("Serious vs Silly mode")
    ax.legend(prop={"family": FONT, "size": 8})
    for t in ax.get_xticklabels() + [ax.title]:
        t.set_fontfamily(FONT)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    display(Image(data=buf.getvalue()))


plot_summary(summary)

wrong = eval_df[(eval_df["type"] == "instruction") & (~eval_df["follow"].astype(bool))]
print(f"沒照做的指令（共 {len(wrong)} 筆）：")
display(wrong[["model", "mode", "cmd", "expect", "got", "raw"]].head(20))

### 6D. 從遊戲 log 計算：通關指令數、延遲、格式正確率、禁區違規

先去 5B / 5C 玩幾關，再回來執行這格。

In [ ]:
def load_play_log(path=LOG_DIR / "game_log.jsonl"):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame([json.loads(line) for line in f if line.strip()])


log_df = load_play_log()
if log_df.empty:
    print("還沒有遊戲紀錄，先去玩幾關吧！")
else:
    llm_turns = log_df[log_df["kind"].isin(["command", "feedback"])]
    print("總回合數：", len(log_df), "｜LLM 回合：", len(llm_turns))
    print(f"格式正確率（第一次就對）：{llm_turns['format_ok'].astype(float).mean():.1%}")
    print(f"可解析率（含重試）：{llm_turns['parse_ok'].astype(float).mean():.1%}")
    print(f"發呆次數：{int(llm_turns['dazed'].sum())}")
    print(f"平均回應延遲：{llm_turns['latency_s'].mean():.2f} 秒")
    with_rule = llm_turns[llm_turns["forbidden_rooms"].map(len) > 0]
    if len(with_rule):
        print(f"記憶遵守率（有禁區時沒闖入的回合比例）：{1 - with_rule['forbidden_violation'].astype(float).mean():.1%}")
    clears = log_df[log_df["level_cleared"]].groupby(["session", "level", "mode"])["n_commands"].min().reset_index()
    if len(clears):
        print("\n通關所需指令數（每次通關取最小值，再依關卡／模式平均）：")
        display(clears.groupby(["level", "mode"])["n_commands"].agg(["mean", "count"]).rename(
            columns={"mean": "平均指令數", "count": "通關次數"}))
    print("\n事件次數統計：")
    print(Counter(t for evs in log_df["events"].dropna() for t in evs).most_common())

### 6E. 試玩問卷（好玩程度 1–5 分）

In [ ]:
# 邀請 5–10 位同學試玩後填寫（1 = 無聊，5 = 超好玩）
SURVEY = {
    # "同學A": {"好玩程度": 5, "最好笑的瞬間": "叫它繞過貓，結果直接吸貓"},
}
if SURVEY:
    scores = [v["好玩程度"] for v in SURVEY.values()]
    print(f"{len(scores)} 位同學，平均好玩程度：{sum(scores) / len(scores):.2f} / 5")
else:
    print("還沒有問卷資料。")

## 7｜常見問題與小提醒

| 發生什麼事？ | 先檢查哪裡？ |
|---|---|
| 顯示「改用離線 FakeLLM」 | 是否使用課程 Kernel、`OPENAI_BASE_URL` / `OPENAI_API_KEY` 是否存在 |
| `Model Not Found` | 1B 列出的模型名稱是否真的存在 |
| 機器人一直「發呆」 | 看 3G 的模型原話：是不是包了一堆解釋文字？可把 `USE_JSON_MODE = True` |
| 即時模式畫面不動 | ipywidgets 前端沒裝好 → 改用 5B 文字模式 |
| 圖上的字是英文 | 伺服器沒有中文字型（不影響遊戲），對話框仍是中文 |
| AI 太蠢過不了關 | 換「正經模式」、降低 `LEVELS[n]["silliness"]`、或用「停」「回充」保底 |
| 延遲太久 | 換小模型（`qwen2.5:3b`）、縮短 `ACTION_SPEC` 範例、`feedback_rounds=0` 關掉事件回饋 |

**一句話總結：** LLM 負責理解與規劃；Python 負責驗證與真的執行；遊戲世界負責回報事件——三者只用 JSON 溝通。

### 之後可以繼續做
- 移植到 pygame：`Game`、`Agent`、`Session` 直接沿用，只要把 `render_png()` 換成 pygame 繪圖，並在遊戲迴圈每個 frame 呼叫 `session.tick()`。
- 加分項目：語音輸入（例如瀏覽器錄音 → Whisper 轉文字 → 丟進 `run_turn`）。
- 增加關卡：在 `LEVELS` 加一個新的地圖字串即可，2C 的檢查會確認灰塵都走得到。

## 🌟 AI 工具使用聲明（學術誠信）

> 依課程規定，請在報告最後寫清楚使用了什麼 AI 工具、如何使用。以下為範例，請依實際情況修改：

1. **使用的 AI 工具：** Claude Code（Anthropic Claude Opus 5.5）
2. **如何使用：** 依據期末專題提案 PDF 與課程 API Quickstart 範本，協助產生本 Notebook 的程式架構
   （等角渲染、遊戲引擎、Prompt／記憶／JSON 驗證、非同步 Session、評估程式）；
   組員再自行測試、調整人設台詞與關卡，並以課程 LLM 實際跑評估數據。